# Extract x1dbk1, x1dtrace, and x1dbk2 from test-data

This notebook builds on `inspiration/extraction_script.py` and focuses on the test data in `test-data/`. It will:

- create `x1d` files if they do not exist
- extract `x1dbk1`, `x1dtrace`, and `x1dbk2` using the same offsets/sizes logic

The trace selection is interactive using `%matplotlib widget` and `ipywidgets`, so the plot should appear inline.

Run the cells top to bottom.

In [7]:
%matplotlib widget

import os
from pathlib import Path
from copy import copy

import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import stistools as stis

# Optional: set CRDS paths if they are not already configured.
# Update CRDS_PATH to match your local cache location if needed.
CRDS_PATH = "/Users/parke/crds_cache"
if "CRDS_PATH" not in os.environ:
    os.environ["CRDS_PATH"] = CRDS_PATH
    os.environ.setdefault("CRDS_SERVER_URL", "https://hst-crds.stsci.edu")
    os.environ.setdefault("iref", f"{CRDS_PATH}/references/hst/iref/")
    os.environ.setdefault("jref", f"{CRDS_PATH}/references/hst/jref/")
    os.environ.setdefault("oref", f"{CRDS_PATH}/references/hst/oref/")
    os.environ.setdefault("lref", f"{CRDS_PATH}/references/hst/lref/")
    os.environ.setdefault("nref", f"{CRDS_PATH}/references/hst/nref/")
    os.environ.setdefault("uref", f"{CRDS_PATH}/references/hst/uref/")

In [2]:
def find_test_data_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for base in candidates:
        td = base / "test-data"
        if td.exists():
            return td
    raise FileNotFoundError("Could not locate test-data directory from current working dir")

TEST_DATA_DIR = find_test_data_dir()
flt_files = sorted(TEST_DATA_DIR.glob("*_flt.fits"))

print(f"Test data dir: {TEST_DATA_DIR}")
print("FLT files:")
for f in flt_files:
    print(f"- {f.name}")

Test data dir: /Users/parke/Repos/airglow/test-data
FLT files:
- of9b05010_flt.fits
- of9b05020_flt.fits
- of9b05030_flt.fits


In [8]:
def get_x1dparams(header) -> dict:
    grating = str(header.get("opt_elem", "")).lower()
    min_params = dict(maxsrch=0.01, bksmode="off")

    if "g140m" in grating:
        newparams = dict(
            extrsize=19,
            bk1offst=-30, bk2offst=30,
            bk1size=20, bk2size=20,
        )
    elif "g140l" in grating:
        newparams = dict(
            extrsize=13,
            bk1offst=-30, bk2offst=30,
            bk1size=20, bk2size=20,
        )
    elif "e140m" in grating:
        newparams = dict(
            extrsize=7,
            bk1size=5, bk2size=5,
        )
    else:
        raise ValueError(f"Unsupported grating: {grating}")

    if "bk1offst" in newparams:
        assert (abs(newparams["bk1offst"]) - newparams["bk1size"] / 2) > (newparams["extrsize"] / 2) + 5
        assert (abs(newparams["bk2offst"]) - newparams["bk2size"] / 2) > (newparams["extrsize"] / 2) + 5

    return {**min_params, **newparams}


def ensure_x1d(fltfile: Path, force: bool = False) -> Path:
    x1dfile = fltfile.with_name(fltfile.name.replace("_flt", "_x1d"))
    if x1dfile.exists() and not force:
        return x1dfile

    header = fits.getheader(fltfile, 0)
    x1d_params = get_x1dparams(header)
    stis.x1d.x1d(str(fltfile), str(x1dfile), **x1d_params)
    return x1dfile


def get_default_traceloc(fltfile: Path) -> float | None:
    header = fits.getheader(fltfile, 0)
    grating = str(header.get("opt_elem", "")).lower()
    x1dfile = ensure_x1d(fltfile, force=False)
    if not x1dfile.exists():
        return None

    x1d = fits.getdata(x1dfile, 1)
    tracelocs = x1d["a2center"]
    if "e140m" in grating:
        return float(tracelocs[-8])
    if np.ndim(tracelocs):
        return float(tracelocs[0])
    return float(tracelocs)


def plot_trace_image(fltfile: Path):
    x1dfile = ensure_x1d(fltfile, force=False)
    data = fits.getdata(fltfile, 1)

    fig, ax = plt.subplots()
    ax.set_title(fltfile.name)
    ax.imshow(np.cbrt(data), aspect="auto")

    if x1dfile.exists():
        x1d = fits.getdata(x1dfile, 1)
        x = np.arange(data.shape[1]) + 0.5
        y = x1d["extrlocy"]
        if np.ndim(y) > 1:
            ax.plot(x, y.T, color="r", lw=0.5, alpha=0.5, label="pipeline trace")
        else:
            ax.plot(x, y, color="r", lw=0.5, alpha=0.5, label="pipeline trace")

    ax.text(0.02, 0.98, "Click the trace. Use the buttons to keep/clear.",
            transform=ax.transAxes, va="top", color="w", fontsize="small")

    if ax.get_legend_handles_labels()[0]:
        ax.legend(loc="lower right")

    return fig, ax


SELECTED_TRACE_Y = None
DEFAULT_TRACE_Y = None


def setup_trace_selector(fltfile: Path):
    global SELECTED_TRACE_Y, DEFAULT_TRACE_Y

    SELECTED_TRACE_Y = None
    DEFAULT_TRACE_Y = get_default_traceloc(fltfile)

    fig, ax = plot_trace_image(fltfile)

    hline = ax.axhline(0, color="y", lw=1, alpha=0.7)
    hline.set_visible(False)

    status = widgets.Label(value="Click on the image to select the trace location.")
    use_default_btn = widgets.Button(description="Use default")
    clear_btn = widgets.Button(description="Clear selection")

    def set_selection(y: float, msg: str) -> None:
        global SELECTED_TRACE_Y
        SELECTED_TRACE_Y = float(y)
        hline.set_ydata([y, y])
        hline.set_visible(True)
        fig.canvas.draw_idle()
        status.value = msg

    def on_click(event):
        if event.inaxes != ax or event.ydata is None:
            return
        set_selection(event.ydata, f"Selected y: {event.ydata:.2f}")

    def use_default(_):
        if DEFAULT_TRACE_Y is None:
            status.value = "Default trace not available."
            return
        set_selection(DEFAULT_TRACE_Y, f"Using default y: {DEFAULT_TRACE_Y:.2f}")

    def clear(_):
        global SELECTED_TRACE_Y
        SELECTED_TRACE_Y = None
        hline.set_visible(False)
        fig.canvas.draw_idle()
        status.value = "Selection cleared."

    fig.canvas.mpl_connect("button_press_event", on_click)
    use_default_btn.on_click(use_default)
    clear_btn.on_click(clear)

    display(widgets.HBox([use_default_btn, clear_btn]), status)
    return fig, ax


def extract_background_traces(
    fltfile: Path,
    overwrite: bool = False,
    manual_traceloc: float | None = None,
) -> None:
    header = fits.getheader(fltfile, 0)
    grating = str(header.get("opt_elem", "")).lower()
    x1d_params = get_x1dparams(header)

    x1dfile = ensure_x1d(fltfile, force=False)
    tracelocs = fits.getdata(x1dfile, 1)["a2center"]

    labels = ["x1dbk1", "x1dtrace", "x1dbk2"]
    mod_params = copy(x1d_params)
    mod_params["bk1size"] = mod_params["bk2size"] = 0
    mod_params["bk1offst"] = mod_params["bk2offst"] = 0
    mod_params.pop("extrsize", None)

    if "e140m" in grating:
        traceloc = manual_traceloc if manual_traceloc is not None else tracelocs[-8]
        sets = ((traceloc, x1d_params["extrsize"], labels[1]),)
    else:
        traceloc = manual_traceloc if manual_traceloc is not None else (tracelocs[0] if np.ndim(tracelocs) else tracelocs)
        y1 = traceloc + x1d_params["bk1offst"]
        yt = traceloc
        y2 = traceloc + x1d_params["bk2offst"]
        sz1 = x1d_params["bk1size"]
        szt = x1d_params["extrsize"]
        sz2 = x1d_params["bk2size"]
        sets = ((y1, sz1, labels[0]), (yt, szt, labels[1]), (y2, sz2, labels[2]))

    for y, sz, lbl in sets:
        out = fltfile.with_name(fltfile.name.replace("_flt", f"_{lbl}"))
        if out.exists() and overwrite:
            out.unlink()
        if out.exists() and not overwrite:
            continue
        stis.x1d.x1d(str(fltfile), str(out), a2center=y, extrsize=sz, **mod_params)

In [9]:
# Pick a single file at a time.
TARGET_FILE = TEST_DATA_DIR / "of9b05010_flt.fits"
print(f"Target: {TARGET_FILE.name}")

Target: of9b05010_flt.fits


In [10]:
# Display the interactive plot and click to select the trace.
# Use the buttons below the plot to use the default or clear the selection.
fig, ax = setup_trace_selector(TARGET_FILE)

/var/folders/6l/1fvgkb5d08b8tf7zpvnzkd7c0000gn/T/ipykernel_41851/4257291237.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  pts = plt.ginput(1, timeout=0)


KeyboardInterrupt: 

In [ ]:
# Run the extractions using the selected trace location.
trace_y = SELECTED_TRACE_Y if SELECTED_TRACE_Y is not None else DEFAULT_TRACE_Y
if trace_y is None:
    raise ValueError("No trace selected and no default trace available.")

print(f"Using trace y: {trace_y}")
extract_background_traces(
    TARGET_FILE,
    overwrite=False,
    manual_traceloc=trace_y,
)

print("\nOutputs:")
for suffix in ("x1dbk1", "x1dtrace", "x1dbk2"):
    out = TARGET_FILE.with_name(TARGET_FILE.name.replace("_flt", f"_{suffix}"))
    print(f"- {out.name} ({'exists' if out.exists() else 'missing'})")